# Analytics Queries
Analysis queries on the gold layer answering key business questions.

In [ ]:
# Top-selling products by revenue (Top N)
N = 10

top_products = spark.sql(f"""
    SELECT ProductName, CategoryName, TotalRevenue, TotalQuantity, OrderCount
    FROM gold_sales_by_product
    ORDER BY TotalRevenue DESC
    LIMIT {N}
""")

print(f"Top {N} Products by Revenue:")
top_products.show(truncate=False)

In [ ]:
# Sales per geography breakdown

geo_sales = spark.sql("""
    SELECT CountryRegion, StateProvince, City, TotalRevenue, OrderCount, CustomerCount
    FROM gold_sales_by_geography
    ORDER BY TotalRevenue DESC
""")

print("Sales by Geography:")
geo_sales.show(30, truncate=False)

# Summary by country
country_summary = spark.sql("""
    SELECT CountryRegion, 
           SUM(TotalRevenue) as TotalRevenue, 
           SUM(OrderCount) as OrderCount,
           SUM(CustomerCount) as CustomerCount
    FROM gold_sales_by_geography
    GROUP BY CountryRegion
    ORDER BY TotalRevenue DESC
""")

print("Summary by Country:")
country_summary.show(truncate=False)

In [ ]:
# Outlier analysis

outliers = spark.sql("""
    SELECT SalesOrderID, TotalDue, MeanOrderValue, StdDev, ZScore,
           CASE WHEN ZScore > 0 THEN 'Above Mean' ELSE 'Below Mean' END as Direction
    FROM gold_sales_outliers
    ORDER BY ABS(ZScore) DESC
""")

print(f"Sales Outliers (beyond 2 std dev): {outliers.count()} orders")
outliers.show(20, truncate=False)

above = outliers.filter("ZScore > 0").count()
below = outliers.filter("ZScore < 0").count()
print(f"Above mean: {above}, Below mean: {below}")

In [ ]:
# Category comparison

categories = spark.sql("""
    SELECT CategoryName,
           SUM(TotalRevenue) as TotalRevenue,
           AVG(TotalRevenue) as AvgProductRevenue,
           COUNT(*) as ProductCount,
           SUM(TotalQuantity) as TotalQuantity
    FROM gold_sales_by_product
    WHERE CategoryName IS NOT NULL
    GROUP BY CategoryName
    ORDER BY TotalRevenue DESC
""")

print("Category Performance Comparison:")
categories.show(truncate=False)